In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

In [4]:
from qiskit import QuantumCircuit
import numpy as np

def qft_rotations(circuit, n):
    """Apply Hadamard and controlled phase rotations recursively."""
    if n == 0:
        return circuit
    n -= 1
    circuit.h(n)
    for qubit in range(n):
        circuit.cp(np.pi / (2 ** (n - qubit)), qubit, n)
    qft_rotations(circuit, n)

def swap_registers(circuit, n):
    """Reverse qubit order using swaps (standard QFT convention)."""
    for qubit in range(n // 2):
        circuit.swap(qubit, n - qubit - 1)
    return circuit

def qft_circuit(n):
    """Build an n-qubit QFT circuit from basic gates."""
    qc = QuantumCircuit(n)
    qft_rotations(qc, n)
    swap_registers(qc, n)
    return qc

def inverse_qft_circuit(n):
    """Build the inverse QFT by inverting the forward QFT."""
    qc = qft_circuit(n)
    return qc.inverse()


# Example: 4-qubit QFT
n = 4
qc = qft_circuit(n)
print("QFT Circuit:")
print(qc.draw())

# Inverse QFT
iqc = inverse_qft_circuit(n)
print("\nInverse QFT Circuit:")
print(iqc.draw())

QFT Circuit:
                                                                          ┌───┐»
q_0: ──────■───────────────────────────────■──────────────────────■───────┤ H ├»
           │                               │                ┌───┐ │P(π/2) └───┘»
q_1: ──────┼────────■──────────────────────┼────────■───────┤ H ├─■─────────X──»
           │        │                ┌───┐ │P(π/4)  │P(π/2) └───┘           │  »
q_2: ──────┼────────┼────────■───────┤ H ├─■────────■───────────────────────X──»
     ┌───┐ │P(π/8)  │P(π/4)  │P(π/2) └───┘                                     »
q_3: ┤ H ├─■────────■────────■─────────────────────────────────────────────────»
     └───┘                                                                     »
«        
«q_0: ─X─
«      │ 
«q_1: ─┼─
«      │ 
«q_2: ─┼─
«      │ 
«q_3: ─X─
«        

Inverse QFT Circuit:
           ┌───┐                                                            »
q_0: ────X─┤ H ├─■────────────────────────■─────────────────────────

In [ ]:

test_qc = QuantumCircuit(n)
test_qc.x(0)  
test_qc.compose(qft_circuit(n), inplace=True)
test_qc.measure_all()

simulator = AerSimulator()
compiled = transpile(test_qc, simulator)
result = simulator.run(compiled, shots=1024).result()
counts = result.get_counts()
print(counts)

{'0010': 62, '1010': 62, '0111': 57, '1000': 65, '1111': 61, '0110': 75, '0011': 55, '0101': 83, '0001': 71, '1110': 62, '1100': 69, '1011': 54, '0000': 68, '1101': 62, '0100': 70, '1001': 48}
